In [1]:
import os


In [2]:
import time


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
import json

class CustomDataset(Dataset):
    def __init__(self, jsonl_files, year, train=True):
        self.data = []
        self.targets = []
        for file in jsonl_files:
            if 'jsonl' not in file:
                continue

            with open(file, 'r') as f:
                for line in f:
                    entry = json.loads(line)
                    for key in entry.keys():
                        if (2013 <= int(key) < year) and train:
                            data_0 = torch.tensor(entry[key]['0'])

                            self.data.append(data_0)
                            self.targets.append(torch.tensor(entry[key]['1']))

                        elif (year <= int(key) <= 2019) and (train is False):
                            data_0 = torch.tensor(entry[key]['0'])

                            self.data.append(data_0)
                            self.targets.append(torch.tensor(entry[key]['1']))

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]


In [4]:
# !pwd


In [5]:
print('Begin')
start_time = time.time()
print(start_time)
jsonl_file_path = 'yearly/Representation/results_mlp_13v_norm/'
jsonl_files_list = os.listdir('yearly/Representation/results_mlp_13v_norm/')
jsonl_files = [jsonl_file_path + x for x in jsonl_files_list]

# 设置年份参数
year = 2019

# 创建训练集和测试集的数据集实例
train_dataset = CustomDataset(jsonl_files, year, train=True)
test_dataset = CustomDataset(jsonl_files, year, train=False)
# todo: save dataset
print('dataset completed. tim gap: ', time.time() - start_time)
start_time = time.time()


Begin
1763100899.858465
dataset completed. tim gap:  14.805744409561157


In [6]:
len(train_dataset.data)


115

In [7]:
train_dataset.data[0].size()


torch.Size([13, 6146])

In [8]:
# Optimized: cache last tensor to avoid double list access
last_tensor = train_dataset.data[-1] if train_dataset.data else None
result = last_tensor[-1] if last_tensor is not None else None
result


tensor([-0.7148, -0.8008,  0.9062,  ...,  0.6250, 12.0000,  0.4814])

In [9]:
len(test_dataset.data)


20

In [10]:
test_dataset.data[0].size()


torch.Size([13, 6146])

In [11]:
len(train_dataset.data)


115

In [12]:
# 保存Dataset对象的状态
torch.save(train_dataset, '../dataset_RT/train_dataset_mlp_13v_id_norm_6146_13-19.csv')
torch.save(test_dataset, '../dataset_RT/test_dataset_mlp_13v_id_norm_6146_13-19.csv')
